# 01. Extracting Invoice Fields with Azure AI Content Understanding

This notebook uses the **Azure AI Content Understanding** service's `prebuilt-invoice` analyzer to extract structured fields (vendor, totals, line items, dates, etc.) from a sample CloudXeus invoice PDF, using the `azure-ai-contentunderstanding` SDK. It belongs to Section 07 of this course, the document/content understanding agent section.

**Difficulty: Intermediate**

Content Understanding is a genuinely AI-102-relevant Azure AI service — it's the newer, more general evolution of the extraction capabilities the exam also covers under Azure AI Document Intelligence's prebuilt models. Prebuilt analyzers like `prebuilt-invoice` are exactly the kind of "use a Microsoft-trained model, no training data of your own required" capability AI-102 tests you on.

## Prerequisites

**pip3 packages:**
```bash
pip3 install azure-ai-contentunderstanding azure-core python-dotenv
```
Neither `azure-ai-contentunderstanding` nor `azure-ai-documentintelligence` are currently in the repo root `requirements.txt` — add whichever this course's scripts actually use (this script imports `azure.ai.contentunderstanding`).

**Azure resources required:**
- An Azure AI Foundry / Azure AI Services multi-service resource with **Content Understanding** enabled, exposing an endpoint of the form `https://<resource>.services.ai.azure.com/`.

**Environment variables** (add to root `.env`):
```
AZURE_CONTENTUNDERSTANDING_ENDPOINT=https://<your-resource>.services.ai.azure.com/
AZURE_CONTENTUNDERSTANDING_KEY=<your-content-understanding-key>
```

**Local file:** this notebook expects `cloudxeus_sample_invoice.pdf` in the **same folder as this notebook** (`07. Section Code/`) — it's already present alongside the original script.

## What You'll Learn

- How to construct a `ContentUnderstandingClient` with an `AzureKeyCredential`
- The `begin_analyze_binary(...)` long-running-operation pattern (submit → poll → `poller.result()`)
- The difference between a `prebuilt-invoice` analyzer (Microsoft-trained, zero-shot) and a custom-trained analyzer
- How to read extracted `markdown` content vs structured `fields` from an analysis result

### Step 1 — Imports, configuration, and the client

`AzureKeyCredential` wraps a plain API key for authentication (as opposed to `DefaultAzureCredential`/Entra ID, used elsewhere in this repo's `11_azure_ai_foundry/` labs). `ContentUnderstandingClient` is constructed once with the service endpoint and that credential.

💡 **Exam tip:** AI-102 expects you to know both authentication styles for Azure AI services — API key (`AzureKeyCredential`, simplest, key must be protected) and Entra ID (`DefaultAzureCredential`, no long-lived secret, supports RBAC) — and to recognize when each is appropriate. Key-based auth like this is fine for a quick script; production services generally prefer Entra ID + managed identity.

🔄 **Alternatives:** swap `AzureKeyCredential(api_key)` for `DefaultAzureCredential()` if your Content Understanding resource is configured for Entra ID auth (requires the caller's identity to have the appropriate Cognitive Services role assigned) — this is the same swap made between key-based and Entra-based clients throughout `11_azure_ai_foundry/`.

In [ ]:
import os
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.ai.contentunderstanding import ContentUnderstandingClient

load_dotenv()

endpoint = os.getenv("AZURE_CONTENTUNDERSTANDING_ENDPOINT")
api_key = os.getenv("AZURE_CONTENTUNDERSTANDING_KEY")

client = ContentUnderstandingClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(api_key),
)


### Step 2 — Submit the invoice for analysis

`begin_analyze_binary(...)` is a long-running-operation (LRO) call: it submits the binary file content for analysis and returns a **poller** immediately, rather than blocking until the whole document is processed. `analyzer_id="prebuilt-invoice"` selects Microsoft's ready-made invoice model — no training required. `poller.result()` blocks and waits for the operation to finish, then returns the final analysis result.

💡 **Exam tip:** the submit-then-poll LRO pattern (`begin_*` methods returning a poller) is pervasive across Azure AI SDKs — Document Intelligence's `begin_analyze_document`, Content Understanding's `begin_analyze_binary`, and Azure AI Search's indexer runs all follow this shape because document analysis can take longer than a typical synchronous HTTP timeout allows. Recognize this pattern name on the exam.

🔄 **Alternatives:** `prebuilt-invoice` works well for standard invoice layouts out of the box. If CloudXeus's invoices had a highly unusual, consistent custom layout that the prebuilt model handled poorly, the AI-102-relevant alternative is training a **custom analyzer** (or, in classic Document Intelligence terms, a custom extraction model) on a labeled set of sample invoices — more setup, but tuned to your exact document layout.

In [ ]:
file_path = "cloudxeus_sample_invoice.pdf"

with open(file_path, "rb") as file:
    poller = client.begin_analyze_binary(
        analyzer_id="prebuilt-invoice",
        binary_input=file,
        content_type="application/pdf",
    )

result = poller.result()
print("Analysis completed.")


### Step 3 — Inspect the extracted markdown

Content Understanding results are keyed under `result["contents"]`, a list with one entry per analyzed document/page grouping — `content = result["contents"][0]` grabs the first (and here, only) one. If the analyzer returned a `markdown` representation of the document (a common Content Understanding output — a readable, structured text rendering of the whole page), this prints the first 1000 characters of it.

💡 **Exam tip:** the `markdown` output is a Content Understanding-specific convenience — it's a normalized textual representation of a document useful for feeding straight into an LLM prompt (as notebook 03 in this series does), distinct from the structured `fields` dictionary used for programmatic field access.

🔄 **Alternatives:** for documents where you only care about structured fields (not a readable text summary), you can skip the markdown inspection entirely and go straight to `content["fields"]`, shown next.

In [ ]:
content = result["contents"][0]

# Print extracted text or markdown if available
print("\n--- Document Content ---")
if "markdown" in content:
    print(content["markdown"][:1000])
else:
    print("No markdown content returned.")


### Step 4 — Inspect extracted structured fields

`content["fields"]` is a dictionary of every field the `prebuilt-invoice` analyzer recognized — vendor name, invoice total, due date, line items, and so on, each with its extracted value and (typically) a confidence score. Looping over `.items()` and printing each pair gives a quick readable dump of everything the model found.

💡 **Exam tip:** know the canonical field set a prebuilt invoice model extracts (vendor/customer name and address, invoice ID, invoice date, due date, total, subtotal, tax, line items) — AI-102 questions sometimes ask you to identify *which* prebuilt model (invoice, receipt, ID document, business card, etc.) is appropriate for a given document type based on this kind of field list.

🔄 **Alternatives:** rather than printing every field, a real application would selectively pull out the fields it needs (e.g., `fields["InvoiceTotal"]`, `fields["VendorName"]`) and validate their confidence scores before trusting them downstream — critical for any workflow (like notebook 03's agent review) that acts on the extracted data automatically.

In [ ]:
fields = content["fields"]

for field_name, field_data in fields.items():
    print(f"{field_name}: {field_data}")


### Step 5 — Close the client

`client.close()` releases the underlying HTTP connection resources. This matters more in long-running processes (a web service handling many requests) than in a short notebook run, but it's good practice to close SDK clients explicitly rather than relying on garbage collection.

💡 **Exam tip:** most Azure SDK clients (including this one) also support use as a context manager (`with ContentUnderstandingClient(...) as client:`), which calls `close()` automatically — either pattern is valid; the explicit close shown here mirrors the original script.

🔄 **Alternatives:** wrap the whole notebook's client usage in a `with` block instead of a manual `.close()` call at the end, for automatic cleanup even if an exception occurs partway through.

In [ ]:
client.close()


## Summary

This notebook submitted a sample invoice PDF to Azure AI Content Understanding's `prebuilt-invoice` analyzer, waited for the long-running analysis to complete via `poller.result()`, then printed both the document's markdown representation and its structured extracted fields (vendor, totals, dates, line items, etc.). The output demonstrates zero-shot structured extraction from an unstructured PDF — no training data or custom model required, because `prebuilt-invoice` is a Microsoft-trained model that already understands common invoice layouts.

## Try It Yourself

1. **Easy:** Print only a subset of fields you care about (e.g., `InvoiceTotal`, `InvoiceDate`, `VendorName`) instead of dumping every field, and format them into a one-line summary string.
2. **Intermediate:** Add a confidence-score check — for any field with a `confidence` value below some threshold (e.g., 0.7), print a warning instead of trusting the value, simulating a real invoice-processing pipeline's validation step.
3. **Advanced:** Try `prebuilt-invoice` against an invoice PDF with a very different layout from the sample (a scanned/photographed invoice, or one in a different language) and compare extraction quality — this is a good way to build intuition for when a custom analyzer would be worth the extra setup cost.